In [2]:
import os
import pandas as pd
import numpy as np
import bintrans
import torch

In [3]:
folder_name = input('模型源文件夹：')

模型源文件夹：model5


归一化的参数特征

In [4]:
param_path = '../../bat_mod/result/param.xlsx'
save_path = '../trained_model/'

In [5]:
scale = 2**12

In [6]:
param = pd.read_excel(param_path, header=None).values
norm_dict = np.load(save_path + folder_name + '/train/norm_dict.npy', allow_pickle=True).item()
maxi = norm_dict['max']
mini = norm_dict['min']
print(maxi)
print(mini)

[4.25000000e+03 1.00000000e+04 1.00000000e+00 4.17762773e+00
 2.84460780e-01 8.92390890e-02 2.11547295e+02 2.11547566e+02]
[2.40000000e+03 0.00000000e+00 0.00000000e+00 3.23357225e+00
 1.49193539e-01 3.16103652e-02 8.28395429e-01 3.47040527e-03]


In [7]:
x_dict = np.load(save_path + folder_name + '/train/o_x_dict.npy', allow_pickle=True).item()
xy_dict = np.load(save_path + folder_name + '/train/xy_dict.npy', allow_pickle=True).item()

In [8]:
#参数特征归一化
param_norm = (param - mini[3:]) / (maxi - mini)[3:]
#量化为整数
param_scal = np.floor(param_norm * scale)
print(param_scal)

[[   0. 4096. 4096. 4096. 4096.]
 [1005.    0.    0.  388.  402.]
 [1387.  921.  674.  549.  563.]
 [1578.  805.  306.  523.  536.]
 [1695.  594.  463.  166.  182.]
 [1863.  581.  386.  119.  135.]
 [2240. 1211.  706. 1283. 1294.]
 [2623.  949.  556.  385.  399.]
 [3058.  533.  572.   16.   32.]
 [3538.  588.  411.  154.  169.]
 [4096.  269.  700.    0.    0.]]


In [9]:
param_scal.shape

(11, 5)

In [10]:
def mat2bintxt(param_scal):
    #将矩阵每一行的所有列合并为字符串
    #每一列为16位2进制数
    #返回列表
    param_list = []
    for i in range(param_scal.shape[0]):
        new_bin = ''
        for j in range(param_scal.shape[1]):
            if (j==(param_scal.shape[1]-1)):
                param_bin = bintrans.dec2bnr(int(param_scal[i, j])) + '\n'
            else:
                param_bin = bintrans.dec2bnr(int(param_scal[i, j]))
            new_bin = new_bin + param_bin
        param_list.append(new_bin)
    return param_list

In [11]:
param_list = mat2bintxt(param_scal)

In [12]:
param_list

['00000000000000000001000000000000000100000000000000010000000000000001000000000000\n',
 '00000011111011010000000000000000000000000000000000000001100001000000000110010010\n',
 '00000101011010110000001110011001000000101010001000000010001001010000001000110011\n',
 '00000110001010100000001100100101000000010011001000000010000010110000001000011000\n',
 '00000110100111110000001001010010000000011100111100000000101001100000000010110110\n',
 '00000111010001110000001001000101000000011000001000000000011101110000000010000111\n',
 '00001000110000000000010010111011000000101100001000000101000000110000010100001110\n',
 '00001010001111110000001110110101000000100010110000000001100000010000000110001111\n',
 '00001011111100100000001000010101000000100011110000000000000100000000000000100000\n',
 '00001101110100100000001001001100000000011001101100000000100110100000000010101001\n',
 '00010000000000000000000100001101000000101011110000000000000000000000000000000000\n']

In [13]:
def bin2coe(blist):
    title = ['memory_initialization_radix=2;\n', 'memory_initialization_vector=\n']
    clist = list(blist)
    for j in range(len(blist)):
        if (j==(len(blist)-1)):
            clist[j] = clist[j].replace('\n', ';\n')
        else:
            clist[j] = clist[j].replace('\n', ',\n')
    return title + clist

In [14]:
bin2coe(param_list)

['memory_initialization_radix=2;\n',
 'memory_initialization_vector=\n',
 '00000000000000000001000000000000000100000000000000010000000000000001000000000000,\n',
 '00000011111011010000000000000000000000000000000000000001100001000000000110010010,\n',
 '00000101011010110000001110011001000000101010001000000010001001010000001000110011,\n',
 '00000110001010100000001100100101000000010011001000000010000010110000001000011000,\n',
 '00000110100111110000001001010010000000011100111100000000101001100000000010110110,\n',
 '00000111010001110000001001000101000000011000001000000000011101110000000010000111,\n',
 '00001000110000000000010010111011000000101100001000000101000000110000010100001110,\n',
 '00001010001111110000001110110101000000100010110000000001100000010000000110001111,\n',
 '00001011111100100000001000010101000000100011110000000000000100000000000000100000,\n',
 '00001101110100100000001001001100000000011001101100000000100110100000000010101001,\n',
 '000100000000000000000001000011010000001010111

In [15]:
with open(save_path + folder_name + '/param/features.txt', 'w') as f:
    f.writelines(param_list)
with open(save_path + folder_name + '/param/features.coe', 'w') as f:
    f.writelines(bin2coe(param_list))

In [42]:
def bin2verilog(blist):
    b_veri = ''
    for i in range(len(blist)):
        b_veri = b_veri + f'16\'d{i}: dout <= 64\'b' + blist[i].replace('\n', ';\n') 
    return b_veri

In [43]:
#创建写入verilog的case
with open(save_path + folder_name + '/param/features_v.txt', 'w') as f:
    f.write(bin2verilog(param_list))

总容量等参数

In [22]:
qmax = 2000
u_min = int(mini[1])
i_min = int(mini[0])
u_max = int(maxi[1])
i_max = int(maxi[0])
u_amp = u_max - u_min
i_amp = i_max - i_min
width = 16

In [23]:
def cal_scale(x):
    i = 0
    while x * (2**i) < 2**(width-1):
        i = i + 1
    scale = i - 1
    return scale

In [24]:
qmax_s = qmax * 3600
qmax_s_rec = 1 / qmax_s
u_amp_rec = 1 / u_amp
i_amp_rec = 1 / i_amp

In [25]:
scale_q = cal_scale(qmax_s_rec)
scale_u = cal_scale(u_amp_rec)
scale_i = cal_scale(i_amp_rec)
print(scale_q)
print(scale_i)
print(scale_u)

37
25
28


In [26]:
u_amp_rec_sca = int(u_amp_rec * (2**scale_u))
i_amp_rec_sca = int(i_amp_rec * (2**scale_i))
qmax_s_rec_sca = int(qmax_s_rec * (2**scale_q))

In [27]:
x_dict['train_x'][0, 0,:]

array([4.20500000e+03, 7.74000000e+02, 9.31645664e-01, 4.04918117e+00,
       1.68634309e-01, 3.74001075e-02, 8.75105893e+00, 8.74939098e+00])

In [28]:
soc_ini = x_dict['train_x'][0, 0, 2]
print(soc_ini)

0.9316456639646234


In [29]:
with open(save_path + folder_name + '/param/parameter.txt', 'w') as file:
    file.write(f'1\
    \n电池容量: {qmax} mah\
    \n更换单位：{qmax_s} mas\
    \n电池容量的倒数：{qmax_s_rec}\
    \n电池容量倒数的放大倍数：{scale_q}\
    \n放大后：{qmax_s_rec_sca}\
    \n2.\
    \n电压落差:{u_amp}\
    \n倒数：{u_amp_rec}\
    \n电压倒数放大倍数:{scale_u}\
    \n放大后：{u_amp_rec_sca}\
    \n3.\
    \n电流落差：{i_amp}\
    \n倒数：{i_amp_rec}\
    \n电流放大倍数:{scale_i}\
    \n放大后：{i_amp_rec_sca}\
    \n4.\
    \n\n电压相反数:{-u_min}\
    \n5.\
    \n电流相反数：{-i_min}\
    \n6.\
    \n初始百分比：{soc_ini}\
    \n容量：{soc_ini * qmax_s}\
              ')

权重与偏置

In [30]:
model = torch.load(save_path + folder_name + '/train/my_model.pth')

In [31]:
parameters = {}
for name, param in model.named_parameters():
    parameters[name] = param.detach().numpy()

In [32]:
l1 = ['U_i1', 'U_f1', 'U_c1', 'U_o1']
l2 = ['V_i1', 'V_f1', 'V_c1', 'V_o1']
l3 = ['U_i2', 'U_f2', 'U_c2', 'U_o2']
l4 = ['V_i2', 'V_f2', 'V_c2', 'V_o2']
l5 = ['b_i1', 'b_f1', 'b_c1', 'b_o1']
l6 = ['b_i2', 'b_f2', 'b_c2', 'b_o2']
l7 = ['fc.weight']
l8 = ['fc.bias']

In [33]:
parameters['U_o1']

array([[-5.37828147e-01,  3.00368220e-01, -1.95220523e-02,
         4.99955744e-01, -1.20127372e-01, -9.46354330e-01,
        -1.78484499e-01, -1.24262078e-02,  1.07314721e-01,
        -3.65183473e-01, -1.41146988e-01, -4.35816824e-01,
        -2.51961559e-01, -1.08160830e+00, -5.85714817e-01,
         2.74243474e-01, -5.96148491e-01, -1.34803265e-01,
        -8.98978040e-02,  8.17414746e-02,  9.72791687e-02,
        -3.64466161e-01, -1.20335169e-01, -1.97026044e-01,
         1.69008151e-01, -2.13899016e-01, -4.87689912e-01,
        -2.27708861e-01, -2.02534217e-02, -2.86543518e-01,
        -3.23046982e-01, -5.17321169e-01, -4.95608896e-02,
        -5.47432482e-01, -8.75206172e-01, -5.21807838e-03,
        -8.34045053e-01, -6.25964940e-01,  2.16007546e-01,
        -3.09217185e-01, -2.86278158e-01,  7.73011297e-02,
         1.01447068e-01, -5.37595391e-01,  1.35449439e-01,
        -2.55149961e-01, -1.69008784e-02, -1.18999019e-01,
        -9.56363752e-02, -8.69296849e-01, -1.09412491e+0

In [34]:
l_dict = {'wx1':l1, 'wh1':l2, 'wx2':l3, 'wh2':l4, 'b1':l5, 'b2':l6, 'wf':l7, 'bf':l8}

In [35]:
#将同一个列表的权重合并
#如4个【64，4】合并为【512，4】矩阵，再合并为【512】的list
new_l = []
for k,u in l_dict.items():
    for i in range(len(u)):
        temp = parameters[u[i]].copy().T.reshape(-1, 1)*scale
        if i==0:
            con = temp.copy()
        else:
            con = np.concatenate((con, temp), axis=1)
    con_lis = mat2bintxt(con)
    new_l.append(con_lis)

In [44]:
i = 0
for k in l_dict.keys():
    filename_t = save_path + folder_name + '/weights/' + k + '.txt'
    filename_c = save_path + folder_name + '/weights/' + k + '.coe'
    filename_v = save_path + folder_name + '/weights/' + k + '_v.txt'
    with open(filename_t, 'w') as f:
        f.writelines(new_l[i])
    with open(filename_c, 'w') as t:       
        t.writelines(bin2coe(new_l[i]))
    with open(filename_v, 'w') as t:       
        t.write(bin2verilog(new_l[i]))
    i = i + 1

输入电流电压文件

In [ ]:
x_list = mat2bintxt(x_dict['train_x'][0, :,:2])

In [ ]:
def bin2hex(binary_string):
    n = len(binary_string) // 4
    hex_string = ''
    for i in range(n):
        start = i * 4
        end = start + 4
        hex_string += hex(int(binary_string[start:end], 2))[2:]
    return hex_string

In [ ]:
x_h_list = []
for i in x_list:
    x_h_list.append(bin2hex(i))

In [ ]:
x_h_list

In [ ]:
with open(save_path + folder_name + '/param/input.txt', 'w') as f:
    f.writelines(x_list)
with open(save_path + folder_name + '/param/input.coe', 'w') as f:
    f.writelines(bin2coe(x_list))

结果对比

In [ ]:
x_in = xy_dict['train_x'][0, :, :]
print(x_in.shape)

In [ ]:
model = torch.load(save_path + folder_name + '/train/my_model.pth')

In [ ]:
# 最终结果
x_ts = torch.from_numpy(x_in.reshape(1, 30, 8)).type(torch.float32)
y = model(x_ts)
print(y)

In [ ]:
# 构造的x
x_ts * scale

In [ ]:
4349 / (2**12)

In [ ]:
x_ts * scale

In [ ]:
x_in